# HSR Minos Demo for Assisted Living Environment

This Jupyter notebook demonstrates how to control a **Toyota Human Support Robot (HSR)** to perform a simple assistive task in an elderly‑care setting. The robot will:

1. Start from its charging station  
2. Greet the elderly person  
3. Navigate to and pick up a water bottle  
4. Deliver the bottle to the elderly person  
5. Return to its home position  


## Safety Warnings

**IMPORTANT** – Before running this program:

* Ensure the robot says **“HSR starto”** before proceeding  
* Release all emergency buttons  
* Make sure the environment is clear of obstacles  
* Always be ready to activate the emergency stop if needed  


## Setting Up Environment

In [ ]:
import os
import hsrb_interface
import rospy
import sys
from hsrb_interface import geometry


## Configuring ROS Network Connection

In [ ]:
# HSR robot acts as ROS master at 'hsrb.local'
# This computer's IP is 10.42.0.1
os.environ['ROS_MASTER_URI'] = 'http://hsrb.local:11311'
os.environ['ROS_IP'] = '10.42.0.1'
print('ROS_MASTER_URI:', os.environ['ROS_MASTER_URI'])
print('ROS_IP:', os.environ['ROS_IP'])


## Defining Robot Constants and Parameters

In [ ]:
# Movement timeout [s]
_MOVE_TIMEOUT = 60.0
# Grasp force [N]
_GRASP_FORCE = 0.2
# TF names of the objects
_BOTTLE_TF = 'ar_marker/4'
_CANE_TF = 'ar_marker/6'
_TOOTHPASTE_TF = 'ar_marker/3'
# TF name of the gripper
_HAND_TF = 'hand_palm_link'


## Initializing Robot Interface

In [ ]:
# Preparation for using robot functions
robot = hsrb_interface.Robot()
omni_base = robot.get('omni_base')
whole_body = robot.get('whole_body')
gripper = robot.get('gripper')
tts = robot.get('default_tts')


## Defining Poses for Object Interaction

In [ ]:
# Pose 0.02 m front & −1.57 rad about z of the bottle marker
bottle_to_hand = geometry.pose(z=-0.05, ek=-1.57)
bottle_to_cane = geometry.pose(z=-0.02, ek=-1.57)
bottle_to_toothpaste = geometry.pose(z=-0.02, ek=-1.57)

# Pose to move the hand 0.1 m upward
hand_up = geometry.pose(x=0.1)

# Pose to move the hand 0.5 m backward
hand_back = geometry.pose(z=-0.5)


## Initial Robot Greeting

In [ ]:
# Greet
whole_body.move_to_go()
whole_body.move_to_neutral()
rospy.sleep(3.0)
whole_body.move_to_go()
tts.say('おはようございます')
rospy.sleep(3.0)


## Setting Up for Bottle Retrieval

In [ ]:
# Transition to initial grasping posture
whole_body.move_to_neutral()
# Keep the hand in view
whole_body.looking_hand_constraint = True
omni_base.go_abs(0, 0, 0, 0.0)


## Navigating to Face Person for Interaction

In [ ]:
whole_body.move_to_go()
omni_base.go_abs(1.095651005508386, 0.880950433864387, 2.878201240685029, 300.0)
whole_body.move_to_neutral()
tts.say('元気ですか？気分はどうですか？何が必要ですか')
whole_body.move_to_joint_positions({'head_tilt_joint': 1.0, 'head_tilt_joint': -0.5})
rospy.sleep(3.0)


## Navigating to Bottle Location

In [ ]:
try:
    whole_body.move_to_go()
    omni_base.go_abs(0.8948337616446661, 1.090646930554913, -0.00487550863654986, 300.0)
    whole_body.move_to_neutral()
    rospy.sleep(1.0)
    whole_body.move_to_go()
    omni_base.go_pose(geometry.pose(z=-1.0, ei=3.14, ej=-1.77), 100.0, ref_frame_id=_BOTTLE_TF)
    tts.say('ボトル持って行きます')
except Exception:
    tts.say('たすけてください')
    rospy.logerr('Failed to navigate to bottle')
    sys.exit()


## Opening Gripper Before Grasping

In [ ]:
gripper.command(1.2)  # 1.2 cm open

## Grasping the Bottle

In [ ]:
try:
    rospy.sleep(2.0)
    whole_body.move_to_neutral()
    whole_body.looking_hand_constraint = True
    whole_body.move_end_effector_pose(bottle_to_hand, _BOTTLE_TF)
    gripper.apply_force(_GRASP_FORCE)
    rospy.sleep(2.0)
    whole_body.move_end_effector_pose(hand_up, _HAND_TF)
    whole_body.move_end_effector_pose(hand_back, _HAND_TF)
    whole_body.move_to_neutral()
except Exception:
    tts.say('たすけてください')
    rospy.logerr('Failed to grasp bottle')
    sys.exit()


## Delivering the Bottle to the Person

In [ ]:
try:
    whole_body.move_to_go()
    omni_base.go_abs(1.095651005508386, 0.880950433864387, 2.878201240685029, 300.0)
    whole_body.move_to_neutral()
    tts.say('どうぞ')
    rospy.sleep(2.0)
    whole_body.looking_hand_constraint = True
    gripper.command(1.2)  # Release
except Exception:
    tts.say('たすけてください')
    rospy.logerr('Failed to deliver bottle')
    sys.exit()


## Returning to Home Position

In [ ]:
whole_body.move_to_go()
omni_base.go_abs(1, 0, 0, 0.0)
omni_base.go_abs(0, 0, 0, 0.0)


## Error Handling

Throughout the notebook, `try/except` blocks ensure safe operation.  
If an exception occurs the robot will:

1. Announce **“たすけてください”** (“Please help me”)  
2. Log the error to ROS  
3. Exit the program  

This guarantees the robot halts safely upon any failure.
